In [5]:
import awswrangler as wr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

GLUE_DATABASE = "classicmodels_star_schema" 
S3_STAGING_DIR = "s3://fgv-datalake-joao-gabriel-9090/athena-results/"

print(f"Bibliotecas carregadas. Conectado ao database: {GLUE_DATABASE}")

Bibliotecas carregadas. Conectado ao database: classicmodels_star_schema


In [6]:
query_1 = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

# 3. Execução via AWS Data Wrangler
print("Enviando query para processamento na AWS Athena...")
try:
    df_products = wr.athena.read_sql_query(
        sql=query_1,
        database=GLUE_DATABASE,
        s3_output=S3_STAGING_DIR
    )
    
    print("\nConsulta finalizada:")
    display(df_products)

except Exception as e:
    print(f"\nErro no Athena: {e}")

Enviando query para processamento na AWS Athena...

Consulta finalizada:


,product_id,product_name,product_line,product_vendor
0,S24_2841,1900s Vintage Bi-Plane,Planes,Autoart Studio Design
1,S24_3432,2002 Chevy Corvette,Classic Cars,Gearbox Collectibles
2,S700_2824,1982 Camaro Z28,Classic Cars,Carousel DieCast Legends
3,S12_3148,1969 Corvair Monza,Classic Cars,Welly Diecast Productions
4,S50_1514,1962 City of Detroit Streetcar,Trains,Classic Metal Creations
5,S32_1374,1997 BMW F650 ST,Motorcycles,Exoto Designs
6,S700_2047,HMS Bounty,Ships,Unimax Art Galleries
7,S32_2206,1982 Ducati 996 R,Motorcycles,Gearbox Collectibles
8,S18_2325,1932 Model A Ford J-Coupe,Vintage Cars,Autoart Studio Design
9,S700_2466,America West Airlines B757-200,Planes,Motor City Art Classics


In [7]:
# Task 4.2:
query_prod = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

print("Executando consulta exploratória...")
df_products = wr.athena.read_sql_query(sql=query_prod, database=GLUE_DATABASE, s3_output=S3_STAGING_DIR)
display(df_products.head())

Executando consulta exploratória...


,product_id,product_name,product_line,product_vendor
0,S18_4522,1904 Buick Runabout,Vintage Cars,Exoto Designs
1,S24_1046,1970 Chevy Chevelle SS 454,Classic Cars,Unimax Art Galleries
2,S700_3167,F/A 18 Hornet 1/72,Planes,Motor City Art Classics
3,S24_1785,1928 British Royal Navy Airplane,Planes,Classic Metal Creations
4,S72_1253,Boeing X-32A JSF,Planes,Motor City Art Classics


In [8]:
# Task 4.3
query_countries = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

print("Calculando Top 10 Países...")
df_countries = wr.athena.read_sql_query(sql=query_countries, database=GLUE_DATABASE, s3_output=S3_STAGING_DIR)
display(df_countries)

Calculando Top 10 Países...


,country,total_sales
0,USA,3273280.05
1,Spain,2198778.18
2,France,1007374.02
3,Singapore,791993.34
4,Australia,562582.59
5,New Zealand,476847.01
6,UK,436947.44
7,Germany,392941.98
8,Italy,360616.81
9,Finland,295149.35


In [9]:
# Task 4.4:
query_base = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

print("Baixando a base analítica para o cache local...")
df_base = wr.athena.read_sql_query(sql=query_base, database=GLUE_DATABASE, s3_output=S3_STAGING_DIR)
df_base['full_date'] = pd.to_datetime(df_base['full_date'])
print(f"Total de linhas na agregação de grão: {len(df_base)}")
display(df_base.head())

Baixando a base analítica para o cache local...
Total de linhas na agregação de grão: 2996


,full_date,product_line,product_name,country,total_sales
0,2004-08-17,Vintage Cars,1938 Cadillac V-16 Presidential Limousine,Italy,1633.05
1,2004-11-15,Classic Cars,1970 Plymouth Hemi Cuda,USA,2448.93
2,2005-02-03,Motorcycles,2002 Suzuki XREO,France,7380.38
3,2004-11-04,Classic Cars,1948 Porsche Type 356 Roadster,USA,3876.60
4,2005-02-23,Vintage Cars,1912 Ford Model T Delivery Wagon,USA,3082.67


In [ ]:
import warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mtick
import seaborn as sns
import ipywidgets as widgets

warnings.filterwarnings('ignore')

# ================
# Styles
# ================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'Arial', 'DejaVu Sans']

plt.rcParams['text.color'] = '#333333'
plt.rcParams['axes.labelcolor'] = '#555555'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlepad'] = 18

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.left'] = False

plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['grid.color'] = '#EAEAEA'
plt.rcParams['grid.linestyle'] = '--'

COLOR_PRIMARY = '#3498db'
COLOR_SECONDARY = '#2c3e50'
COLOR_STATIC = '#bdc3c7'

COLOR_BG_CARD = '#f8f9fa'
COLOR_BORDER = '#dee2e6'

# ================
# filters
# ================
df_static_prod = (
    df_base.groupby('product_name')['total_sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

df_static_country = (
    df_base.groupby('country')['total_sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

min_date = df_base['full_date'].min().date()
max_date = df_base['full_date'].max().date()

lista_paises = ['Todos'] + sorted(df_base['country'].unique().tolist())
lista_linhas = ['Todas'] + sorted(df_base['product_line'].unique().tolist())

w_start_date = widgets.DatePicker(
    description='Start',
    value=min_date
)

w_end_date = widgets.DatePicker(
    description='End',
    value=max_date
)

w_country = widgets.Dropdown(
    options=lista_paises,
    value='Todos',
    description='Country'
)

w_line = widgets.Dropdown(
    options=lista_linhas,
    value='Todas',
    description='Line'
)

w_top_n = widgets.IntSlider(
    value=5,
    min=1,
    max=15,
    step=1,
    description='Top N'
)

# ================
# header (just a simple css)
# ================
header_html = widgets.HTML(
    value="""
    <div style="
        background-color:#2c3e50;
        padding:12px 20px;
        border-radius:6px;
        margin-bottom:12px;
    ">
        <h2 style="
            color:white;
            margin:0;
            font-family:Segoe UI;
        ">
            Executive Sales Dashboard
        </h2>

        <p style="
            color:#bdc3c7;
            margin-top:4px;
            font-size:12px;
        ">
            Dynamic analytical view
        </p>
    </div>
    """
)

# ================
# to not appear +1e6, or +1e7
# ================
def millions(x, pos):
    return f'{x/1_000_000:.1f}M'

def update_dashboard(start_date, end_date, country, line, top_n):

    if not start_date or not end_date:
        return

    mask = (
        (df_base['full_date'].dt.date >= start_date) &
        (df_base['full_date'].dt.date <= end_date)
    )

    df_filtrado = df_base[mask]

    if country != 'Todos':
        df_filtrado = df_filtrado[
            df_filtrado['country'] == country
        ]

    if line != 'Todas':
        df_filtrado = df_filtrado[
            df_filtrado['product_line'] == line
        ]

    df_top = (
        df_filtrado.groupby('product_name')['total_sales']
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
        .reset_index()
    )

    df_trend = df_filtrado.copy()

    df_trend['month_year'] = (
        df_trend['full_date']
        .dt.strftime('%Y-%m')
    )

    df_trend = (
        df_trend.groupby('month_year')['total_sales']
        .sum()
        .reset_index()
        .sort_values('month_year')
    )

    faturamento_total = df_filtrado['total_sales'].sum()
    faturamento_milhoes = faturamento_total / 1_000_000

    produtos_distintos = df_filtrado['product_name'].nunique()

    fig = plt.figure(
        figsize=(18, 16),
        facecolor='white'
    )

    gs = gridspec.GridSpec(
        3,
        2,
        height_ratios=[1.1, 1.6, 1.1],
        hspace=0.65,
        wspace=0.45
    )

    ax_top = fig.add_subplot(gs[0, 0])

    ax_top.grid(axis='x')

    if not df_top.empty:
        sns.barplot(
            data=df_top,
            x='total_sales',
            y='product_name',
            color=COLOR_PRIMARY,
            ax=ax_top
        )

    ax_top.set_title(
        f'Top {top_n} Products',
        loc='left',
        color=COLOR_SECONDARY
    )

    ax_top.set_xlabel('Sales (US$)')
    ax_top.set_ylabel('')

    ax_kpi = fig.add_subplot(gs[0, 1])

    ax_kpi.axis('off')

    box_style = dict(
        boxstyle='round,pad=1.5',
        facecolor=COLOR_BG_CARD,
        edgecolor=COLOR_BORDER,
        lw=1.5
    )

    ax_kpi.text(
        0.5,
        0.72,
        f"TOTAL REVENUE\n\nUS$ {faturamento_milhoes:,.2f}M",
        fontsize=15,
        ha='center',
        va='center',
        fontweight='bold',
        color=COLOR_SECONDARY,
        bbox=box_style
    )

    ax_kpi.text(
        0.5,
        0.25,
        f"DISTINCT PRODUCTS\n\n{produtos_distintos}",
        fontsize=15,
        ha='center',
        va='center',
        fontweight='bold',
        color=COLOR_SECONDARY,
        bbox=box_style
    )

    ax_trend = fig.add_subplot(gs[1, :])

    ax_trend.grid(axis='y')

    if not df_trend.empty:

        ax_trend.fill_between(
            df_trend['month_year'],
            df_trend['total_sales'],
            color=COLOR_PRIMARY,
            alpha=0.12
        )

        sns.lineplot(
            data=df_trend,
            x='month_year',
            y='total_sales',
            marker='o',
            linewidth=3,
            markersize=8,
            color=COLOR_PRIMARY,
            ax=ax_trend
        )

    ax_trend.set_title(
        'Monthly Revenue Trend',
        loc='left',
        color=COLOR_SECONDARY
    )

    ax_trend.set_xlabel('')
    ax_trend.set_ylabel('Sales (US$)')

    ax_trend.tick_params(
        axis='x',
        rotation=45
    )

    ax_stat_prod = fig.add_subplot(gs[2, 0])

    ax_stat_prod.grid(axis='x')

    sns.barplot(
        data=df_static_prod,
        x='total_sales',
        y='product_name',
        color=COLOR_STATIC,
        ax=ax_stat_prod
    )

    ax_stat_prod.xaxis.set_major_formatter(
        mtick.FuncFormatter(millions)
    )

    ax_stat_prod.set_title(
        'Historical Top Products',
        loc='left',
        fontsize=12,
        color='#7f8c8d'
    )

    ax_stat_prod.set_xlabel('Sales (Million US$)')
    ax_stat_prod.set_ylabel('')

    ax_stat_country = fig.add_subplot(gs[2, 1])

    ax_stat_country.grid(axis='x')

    sns.barplot(
        data=df_static_country,
        x='total_sales',
        y='country',
        color=COLOR_STATIC,
        ax=ax_stat_country
    )

    ax_stat_country.xaxis.set_major_formatter(
        mtick.FuncFormatter(millions)
    )

    ax_stat_country.set_title(
        'Historical Top Countries',
        loc='left',
        fontsize=12,
        color='#7f8c8d'
    )

    ax_stat_country.set_xlabel('Sales (Million US$)')
    ax_stat_country.set_ylabel('')

    plt.tight_layout(pad=3)
    plt.show()

out = widgets.interactive_output(
    update_dashboard,
    {
        'start_date': w_start_date,
        'end_date': w_end_date,
        'country': w_country,
        'line': w_line,
        'top_n': w_top_n
    }
)

box_filtros = widgets.VBox([
    widgets.HBox(
        [w_start_date, w_end_date],
        layout=widgets.Layout(margin='0 0 12px 0')
    ),

    widgets.HBox(
        [w_country, w_line, w_top_n],
        layout=widgets.Layout(margin='0 0 18px 0')
    )
])

ui_final = widgets.VBox([
    header_html,
    box_filtros
])

display(ui_final, out)

Output()